# 4. Transformación Silver – Productos

**Objetivo:** leer `bronze.productos`, separar los cinco registros que tienen `costo_unitario` nulo y preparar dos tablas Delta:

- `silver.productos`: productos con costo unitario informado.
- `silver.productos_rechazados`: productos excluidos por no tener costo unitario, conservando sus datos y el motivo del rechazo.

**Regla acordada como propuesta de tratamiento:** no inventar ni completar los costos faltantes. Los registros con `costo_unitario` nulo se envían a cuarentena.

La tabla Bronze no se modifica. Este notebook define sus propios parámetros y no depende de variables de otros notebooks.

## 1. Inicialización

In [ ]:
dbutils.widgets.removeAll()

In [ ]:
from pyspark.sql.functions import col, lit, current_timestamp

## 2. Parámetros

In [ ]:
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")

containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

esquemaBronze = "bronze"
esquemaSilver = "silver"

ruta_silver = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/silver/"

tabla_productos_bronze = f"{catalogo}.{esquemaBronze}.productos"
tabla_productos_silver = f"{catalogo}.{esquemaSilver}.productos"
tabla_productos_rechazados = f"{catalogo}.{esquemaSilver}.productos_rechazados"

print("Catálogo:", catalogo)
print("Origen:", tabla_productos_bronze)
print("Destino:", tabla_productos_silver)
print("Rechazados:", tabla_productos_rechazados)
print("Ruta Silver:", ruta_silver)

## 3. Lectura desde Bronze

In [ ]:
df_productos_bronze = spark.table(tabla_productos_bronze)

print("Registros en Bronze:", df_productos_bronze.count())
df_productos_bronze.printSchema()

## 4. Separación de registros con costo unitario nulo

Se consideran rechazados los productos cuyo campo `costo_unitario` es NULL. No se reemplaza el costo por cero ni por un valor estimado.

La separación se realiza en memoria; todavía no se escribe ninguna tabla.

In [ ]:
df_productos_rechazados = (
    df_productos_bronze
    .filter(col("costo_unitario").isNull())
    .withColumn("motivo_rechazo", lit("COSTO_UNITARIO_NULO"))
    .withColumn("fecha_proceso", current_timestamp())
)

df_productos_silver = (
    df_productos_bronze
    .filter(col("costo_unitario").isNotNull())
)

print("Productos Bronze:", df_productos_bronze.count())
print("Productos Silver:", df_productos_silver.count())
print("Productos rechazados:", df_productos_rechazados.count())

## 5. Validación de conteos y valores nulos

In [ ]:
conteo_bronze = df_productos_bronze.count()
conteo_silver = df_productos_silver.count()
conteo_rechazados = df_productos_rechazados.count()

print("Bronze:", conteo_bronze)
print("Silver:", conteo_silver)
print("Rechazados:", conteo_rechazados)
print("Conciliación:", conteo_silver + conteo_rechazados)

assert conteo_bronze == conteo_silver + conteo_rechazados, \
    "Los conteos de Bronze, Silver y rechazados no concilian."

assert conteo_rechazados == 5, \
    "Se esperaban 5 productos con costo_unitario nulo."

assert conteo_silver == 995, \
    "Se esperaban 995 productos con costo_unitario informado."

nulos_costo_bronze = df_productos_bronze.filter(col("costo_unitario").isNull()).count()
nulos_costo_silver = df_productos_silver.filter(col("costo_unitario").isNull()).count()

print("Costos nulos en Bronze:", nulos_costo_bronze)
print("Costos nulos en Silver:", nulos_costo_silver)

assert nulos_costo_bronze == 5, \
    "La cantidad de costos nulos en Bronze no coincide con el diagnóstico."

assert nulos_costo_silver == 0, \
    "Silver todavía contiene costos unitarios nulos."

print("Validaciones correctas.")

## 6. Inspección antes de escribir

In [ ]:
display(df_productos_silver.orderBy("producto_id").limit(10))

In [ ]:
display(df_productos_rechazados.orderBy("producto_id"))

## 7. Preparar esquema Silver

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquemaSilver}")
print(f"Esquema disponible: {catalogo}.{esquemaSilver}")

## 8. Escritura de tablas Delta

Ejecutar estas celdas solo después de revisar las validaciones y la inspección.

El modo `overwrite` reconstruye las tablas Silver de Productos si se vuelve a ejecutar el proceso. No modifica Bronze.

In [ ]:
df_productos_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{ruta_silver}productos/") \
    .saveAsTable(tabla_productos_silver)

print("Tabla creada:", tabla_productos_silver)

In [ ]:
df_productos_rechazados.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{ruta_silver}productos_rechazados/") \
    .saveAsTable(tabla_productos_rechazados)

print("Tabla creada:", tabla_productos_rechazados)

## 9. Validación final

In [ ]:
print("Productos Silver:", spark.table(tabla_productos_silver).count())
print("Productos rechazados:", spark.table(tabla_productos_rechazados).count())

spark.table(tabla_productos_silver).printSchema()